In [ ]:
import csv, gzip, hashlib, importlib.util, json, re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config/public-service-keywords-v1.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/public-service-keywords-v1.json").exists(), "Run from project or notebooks directory"
OUT = ROOT / "data/interim/national-service-coverage-audit-v1"
PROD = ROOT / "data/processed/public-service-keywords-v1"
def read_csv(path):
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
places = read_csv(PROD / "places.csv")
excluded = read_csv(PROD / "matched-exclusions.csv")
run = json.loads((PROD / "summary.json").read_text())
selected_ids = {r["gmap_id"] for r in places}
assert len(selected_ids) == len(places) == run["primary_unique_places"]
primary = Counter(r["service_family"] for r in places)
multilabel = Counter(f for r in places for f in set(r["all_service_families"].split(";")))
irs = re.compile(r"\b(?:internal revenue service|irs)\b", re.I)
irs_count = sum(bool(irs.search(r["name"])) for r in places)
ca_dmv = sum("California" in r["source_partitions"].split(";")
             and "motor_vehicles" in r["all_service_families"].split(";") for r in places)
audit = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "production_completed_at_utc": run["completed_at_utc"],
    "selected_unique_gmap_ids": len(places),
    "partitions_processed": len(run["partitions"]),
    "primary_counts": dict(primary), "multilabel_counts": dict(multilabel),
    "irs_name_listings_not_confirmed_tacs": irs_count,
    "california_dmv_listings_not_confirmed_government_offices": ca_dmv,
    "excluded_archive_rows": len(excluded),
    "excluded_primary_family_rows": dict(Counter(r["service_family"] for r in excluded)),
    "input_hashes": {p.name: sha256(p) for p in
        [PROD / "places.csv", PROD / "matched-exclusions.csv",
         ROOT / "config/public-service-keywords-v1.json"]},
}
print(json.dumps(audit, ensure_ascii=False, indent=2))


In [ ]:
states = set("AL AK AZ AR CA CO CT DE DC FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY".split())
roster_path = ROOT / "data/external/official-rosters/processed/usgs-national-structures-current.csv.gz"
types = {"Post Offices", "Courthouses", "Police Stations", "City/Town Halls"}
all_counts, us_counts, other_states, vintages = Counter(), Counter(), Counter(), Counter()
keys = set()
with gzip.open(roster_path, "rt", encoding="utf-8") as handle:
    for r in csv.DictReader(handle):
        if r["source_subtype"] not in types:
            continue
        key = (r["source_subtype"], r["source_record_id"])
        assert key not in keys, "Duplicate ID within a USGS layer"
        keys.add(key)
        all_counts[r["source_subtype"]] += 1
        vintages[r["source_vintage"]] += 1
        if r["state"] in states:
            us_counts[r["source_subtype"]] += 1
        else:
            other_states[r["state"]] += 1
audit["usgs"] = {
    "all_territories_counts": dict(all_counts),
    "fifty_states_dc_counts": dict(us_counts),
    "excluded_territory_counts": dict(other_states),
    "snapshot_vintages": dict(vintages),
    "grain": "named_structure_point_not_proof_of_public_access",
    "sha256": sha256(roster_path),
}
examples = []
for status, records in [("excluded", excluded), ("selected", places)]:
    for r in records:
        if any(n.casefold() in r["name"].casefold() for n in [
            "Apache Junction Municipal Court", "Autauga County Probate Court",
            "Registration Express Group REDDING"]):
            examples.append({k: r[k] for k in [
                "gmap_id", "name", "categories", "service_family", "exclusion_reason"]} | {"status": status})
audit["diagnostic_examples"] = examples
print(json.dumps({"usgs": audit["usgs"], "examples": examples}, ensure_ascii=False, indent=2))


In [ ]:
patterns = {
    "court_alias": r"\b(?:justice court|justice of the peace|judicial center|judicial building|court house)\b",
    "uscis": r"\b(?:uscis|u\.?s\.? citizenship and immigration services|united states citizenship and immigration services)\b",
    "housing_authority": r"\bhousing authority\b",
    "veterans_office": r"\b(?:veterans? (?:affairs|services|benefits)|va regional (?:office|benefit))\b",
    "workforce_brands": r"\b(?:careersource|careerlink|worksource|workforce solutions|one.stop career center)\b",
    "dmv_state_alias": r"\b(?:secretary of state|department of public safety|driver services facility)\b",
    "tax_alias": r"\b(?:tax commissioner|taxpayer assistance center|irs)\b",
}
compiled = {k: re.compile(v, re.I) for k, v in patterns.items()}
gate = re.compile("|".join("(?:%s)" % p for p in patterns.values()), re.I)
spec = importlib.util.spec_from_file_location("extractor", ROOT / "scripts/13_extract_public_service_places.py")
extractor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(extractor)
rules = extractor.load_rules(ROOT / "config/public-service-keywords-v1.json")
inputs = [ROOT / "data/raw/ucsd-google-local" / ("meta-%s.json.gz" % p) for p in extractor.PARTITIONS]
assert len(inputs) == 52 and all(p.exists() for p in inputs)
probe_rows = {}
records_seen = 0
for path in inputs:
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            records_seen += 1
            r = json.loads(line)
            name = str(r.get("name") or "")
            gid = r.get("gmap_id")
            if not gid or gid in selected_ids or not gate.search(name):
                continue
            hits = [k for k, p in compiled.items() if p.search(name)]
            families, _, reasons = extractor.classify(r, rules)
            if gid not in probe_rows:
                probe_rows[gid] = {
                    "gmap_id": gid, "name": name, "address": r.get("address") or "",
                    "categories": " | ".join(r.get("category") or []),
                    "probe_patterns": ";".join(hits),
                    "v1_families": ";".join(families), "v1_exclusions": ";".join(reasons),
                    "partition": path.name[5:-8],
                    "interpretation": "potential_gap_not_confirmed_eligible",
                }
            else:
                old = probe_rows[gid]
                old["probe_patterns"] = ";".join(sorted(set(old["probe_patterns"].split(";")) | set(hits)))
assert records_seen == run["totals"]["records_seen"]
probe_counts = Counter(k for r in probe_rows.values() for k in r["probe_patterns"].split(";"))
audit["raw_probe"] = {
    "records_seen": records_seen,
    "partitions_seen": len(inputs),
    "unique_unselected_name_probe_ids": len(probe_rows),
    "counts_by_probe_not_confirmed_omissions": dict(probe_counts),
    "patterns": patterns,
    "caveat": "Keyword hits can be out of scope. These are not validated missing government offices.",
}
OUT.mkdir(parents=True, exist_ok=True)
with (OUT / "potential-gaps.csv").open("w", encoding="utf-8", newline="") as handle:
    fields = ["gmap_id", "name", "address", "categories", "probe_patterns",
              "v1_families", "v1_exclusions", "partition", "interpretation"]
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(sorted(probe_rows.values(), key=lambda r: r["gmap_id"]))
print(json.dumps(audit["raw_probe"], ensure_ascii=False, indent=2))
for k in patterns:
    examples = [r["name"] for r in probe_rows.values() if k in r["probe_patterns"].split(";")][:5]
    print(k, json.dumps(examples, ensure_ascii=False))


In [ ]:
audit["official_reference_values"] = [
    {"source": "USPS FY2021", "value": 31247, "unit": "managed offices_stations_branches_including_annexes", "period": "FY2021"},
    {"source": "SSA open data", "value": 1230, "unit": "approximate_field_offices", "period": "page_updated_2024-03-08"},
    {"source": "IRS Taxpayer Advocate", "value": 358, "unit": "taxpayer_assistance_centers", "period": "2021-03"},
    {"source": "California DMV", "value": 168, "unit": "field_offices", "period": "2025"},
    {"source": "BJS CSLLEA", "value": 17541, "unit": "state_local_law_enforcement_agencies_not_stations", "period": "2018"},
]
audit["limits"] = [
    "No denominator is established for 2021 nationwide eligible public-facing service locations.",
    "No office-level official-roster matching was run in this audit.",
    "USGS structure points do not prove operation, government ownership or public access.",
    "A gmap_id is a listing, not a deduplicated physical government office.",
    "External counts differ in date, geography, ownership and entity grain.",
    "No recall, precision, missed-office count or population satisfaction estimate is established.",
    "Production rules and selected datasets remain unchanged.",
]
with (OUT / "audit-summary.json").open("w", encoding="utf-8") as handle:
    json.dump(audit, handle, ensure_ascii=False, indent=2)
print("Audit complete. Production classifications unchanged.")
print("Summary:", (OUT / "audit-summary.json").relative_to(ROOT))
print("Potential gaps:", (OUT / "potential-gaps.csv").relative_to(ROOT))
